# MobileNetV3 Age Estimation - RMSE Optimization

**Goal:** Optimize MobileNetV3-small model for age estimation using RMSE as the primary evaluation metric.

**Experiment Focus:**
- Train MobileNetV3-small for age regression on UTKFace dataset
- Use **Root Mean Square Error (RMSE)** as the optimization target
- Comprehensive RMSE evaluation and analysis
- Model performance visualization and comparison

**Key Features:**
- RMSE-focused training pipeline with early stopping
- Detailed RMSE evaluation across age groups
- Performance visualization and error analysis
- Best model selection based on validation RMSE

**Dataset:** UTKFace - 48,212 face images with age labels (1-116 years)

**Model:** MobileNetV3-small with custom regression head for age prediction

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully")
print("Key libraries: PyTorch, torchvision, PIL, numpy, pandas, matplotlib")

## 2. Device Configuration

In [ ]:
# Setup device for training (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
if device.type == 'cuda':
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
    
print(f"   PyTorch Version: {torch.__version__}")
print("Device configuration complete")

## 3. Image Preprocessing Configuration

In [ ]:
# ImageNet normalization values for pretrained models
INPUT_SIZE = 224
MEAN = [0.485, 0.456, 0.406]  # ImageNet standard
STD = [0.229, 0.224, 0.225]   # ImageNet standard

# Training transforms with data augmentation
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Validation transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

print("Image preprocessing configured")
print(f"   Input size: {INPUT_SIZE}x{INPUT_SIZE}")
print(f"   Training augmentations: crop, flip, color jitter, rotation")
print(f"   Normalization: ImageNet standard")

## 4. UTKFace Dataset Loading

In [ ]:
class UTKFaceDataset(Dataset):
    """UTKFace dataset for age estimation training and evaluation"""
    
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.samples = []
        
        print(f"Loading UTKFace dataset from: {data_dir}")
        
        # Load images from part1, part2, part3 directories
        for subdir in ['part1', 'part2', 'part3']:
            subdir_path = self.data_dir / subdir
            if subdir_path.exists():
                print(f"   Processing {subdir}...")
                for img_path in subdir_path.glob("*.jpg"):
                    try:
                        # Extract age from filename: [age]_[gender]_[race]_[timestamp].jpg
                        age = int(img_path.name.split('_')[0])
                        if 0 <= age <= 100:  # Filter reasonable ages
                            self.samples.append((str(img_path), float(age)))
                    except (ValueError, IndexError):
                        continue  # Skip malformed filenames
            else:
                print(f"   Warning: {subdir} directory not found")
        
        if not self.samples:
            raise ValueError(f"No valid samples found in {data_dir}")
        
        # Dataset statistics
        ages = [s[1] for s in self.samples]
        print(f"\nDataset Statistics:")
        print(f"   Total samples: {len(self.samples):,}")
        print(f"   Age range: {min(ages):.0f} - {max(ages):.0f} years")
        print(f"   Mean age: {np.mean(ages):.1f} ± {np.std(ages):.1f} years")
        print(f"   Age distribution: {np.percentile(ages, [25, 50, 75])}")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, age = self.samples[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, torch.tensor(age, dtype=torch.float32)
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a zero tensor as fallback
            if self.transform:
                dummy_image = self.transform(Image.new('RGB', (224, 224), color=(128, 128, 128)))
            else:
                dummy_image = torch.zeros(3, 224, 224)
            return dummy_image, torch.tensor(0.0, dtype=torch.float32)

print("UTKFaceDataset class defined")

In [ ]:
# Load dataset and create data loaders
UTKFACE_PATH = "data/utkface_cropped"  # Adjust path as needed
BATCH_SIZE = 32
NUM_WORKERS = 0  # Use 0 for Windows compatibility

if os.path.exists(UTKFACE_PATH):
    print(f"Loading UTKFace dataset from: {UTKFACE_PATH}")
    
    # Create full dataset
    full_dataset = UTKFaceDataset(UTKFACE_PATH, train_transform)
    
    # Split dataset (80% train, 20% validation)
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    
    train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset, [train_size, val_size], 
        generator=torch.Generator().manual_seed(42)  # Reproducible split
    )
    
    # Apply different transforms to validation set
    val_dataset.dataset.transform = val_transform
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available()
    )
    
    print(f"\nData Split:")
    print(f"   Training samples: {len(train_dataset):,}")
    print(f"   Validation samples: {len(val_dataset):,}")
    print(f"   Batch size: {BATCH_SIZE}")
    print(f"   Training batches: {len(train_loader)}")
    print(f"   Validation batches: {len(val_loader)}")
    print("Data loaders ready")
    
else:
    print(f"Error: UTKFace dataset not found at: {UTKFACE_PATH}")
    print("   Please ensure the dataset is placed with part1, part2, part3 subdirectories")
    raise FileNotFoundError(f"Dataset not found at {UTKFACE_PATH}")

## 5. MobileNetV3 Age Model Creation

In [ ]:
def create_mobilenetv3_age_model(dropout_rate=0.3, hidden_size=128):
    """
    Create MobileNetV3-small model for age regression
    
    Args:
        dropout_rate: Dropout probability for regularization
        hidden_size: Size of hidden layer in classifier
    
    Returns:
        MobileNetV3-small model configured for age estimation
    """
    print(f"Creating MobileNetV3-small model for age regression...")
    
    # Load pretrained MobileNetV3-small
    model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    
    # Get the input features of the original classifier
    num_features = model.classifier[0].in_features
    print(f"   Original classifier input features: {num_features}")
    
    # Replace classifier with age regression head
    model.classifier = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(num_features, hidden_size),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_rate * 0.7),  # Slightly less dropout in second layer
        nn.Linear(hidden_size, 64),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_rate * 0.5),  # Even less dropout in final layer
        nn.Linear(64, 1),
        nn.ReLU()  # Ensure positive age predictions
    )
    
    # Move to device
    model = model.to(device)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"   Model created successfully")
    print(f"   Total parameters: {total_params:,}")
    print(f"   Trainable parameters: {trainable_params:,}")
    print(f"   Model configured for age regression (0-100+ years)")
    
    return model

# Create the model
model = create_mobilenetv3_age_model(dropout_rate=0.3, hidden_size=128)

# Display model architecture summary
print(f"\nModel Architecture Summary:")
print(f"   Backbone: MobileNetV3-small (pretrained)")
print(f"   Input: 3 x 224 x 224 RGB images")
print(f"   Output: Single age prediction (continuous value)")
print(f"   Classifier layers: 3 fully connected layers with ReLU and Dropout")
print("MobileNetV3 age estimation model ready")

## 6. RMSE Metric Implementation

In [ ]:
def calculate_rmse(predictions, targets):
    """
    Calculate Root Mean Square Error between predictions and targets
    
    Args:
        predictions: Model predictions (tensor, numpy array, or list)
        targets: Ground truth values (tensor, numpy array, or list)
    
    Returns:
        RMSE value as float
    """
    # Convert to numpy arrays
    if torch.is_tensor(predictions):
        predictions = predictions.detach().cpu().numpy()
    elif isinstance(predictions, list):
        predictions = np.array(predictions)
    
    if torch.is_tensor(targets):
        targets = targets.detach().cpu().numpy()
    elif isinstance(targets, list):
        targets = np.array(targets)
    
    mse = np.mean((predictions - targets) ** 2)
    rmse = np.sqrt(mse)
    return rmse

def calculate_mae(predictions, targets):
    """Calculate Mean Absolute Error for comparison"""
    # Convert to numpy arrays
    if torch.is_tensor(predictions):
        predictions = predictions.detach().cpu().numpy()
    elif isinstance(predictions, list):
        predictions = np.array(predictions)
    
    if torch.is_tensor(targets):
        targets = targets.detach().cpu().numpy()
    elif isinstance(targets, list):
        targets = np.array(targets)
    
    mae = np.mean(np.abs(predictions - targets))
    return mae

def rmse_loss_function(predictions, targets):
    """
    RMSE loss function for PyTorch training
    
    Args:
        predictions: Model output tensor
        targets: Ground truth tensor
    
    Returns:
        RMSE loss tensor
    """
    mse = torch.mean((predictions - targets) ** 2)
    rmse = torch.sqrt(mse + 1e-8)  # Add small epsilon for numerical stability
    return rmse

def evaluate_model_rmse(model, dataloader, device):
    """
    Evaluate model performance using RMSE metric
    
    Args:
        model: Trained PyTorch model
        dataloader: DataLoader for evaluation
        device: PyTorch device
    
    Returns:
        Dictionary with RMSE, MAE, and detailed metrics
    """
    model.eval()
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for images, ages in dataloader:
            images, ages = images.to(device), ages.to(device)
            predictions = model(images).squeeze()
            
            all_predictions.append(predictions.cpu())
            all_targets.append(ages.cpu())
    
    # Combine all predictions and targets
    all_predictions = torch.cat(all_predictions)
    all_targets = torch.cat(all_targets)
    
    # Calculate metrics
    rmse = calculate_rmse(all_predictions, all_targets)
    mae = calculate_mae(all_predictions, all_targets)
    
    # Additional statistics
    errors = np.abs(all_predictions.numpy() - all_targets.numpy())
    
    results = {
        'rmse': rmse,
        'mae': mae,
        'mean_error': np.mean(errors),
        'std_error': np.std(errors),
        'median_error': np.median(errors),
        'max_error': np.max(errors),
        'predictions': all_predictions.numpy(),
        'targets': all_targets.numpy(),
        'errors': errors
    }
    
    return results

print("RMSE metric functions defined:")
print("   calculate_rmse() - Basic RMSE calculation")
print("   calculate_mae() - Mean Absolute Error for comparison")
print("   rmse_loss_function() - PyTorch RMSE loss for training")
print("   evaluate_model_rmse() - Comprehensive model evaluation")

## 7. Training Pipeline with RMSE Tracking

In [ ]:
def train_mobilenetv3_rmse(model, train_loader, val_loader, epochs=50, lr=0.001, 
                          patience=15, save_path="models/best_mobilenetv3_rmse.pth"):
    """
    Train MobileNetV3 model optimized for RMSE performance
    
    Args:
        model: MobileNetV3 model
        train_loader: Training data loader
        val_loader: Validation data loader
        epochs: Maximum number of training epochs
        lr: Learning rate
        patience: Early stopping patience
        save_path: Path to save best model
    
    Returns:
        Dictionary with training history
    """
    print(f"Starting RMSE-optimized training...")
    print(f"   Epochs: {epochs} (max)")
    print(f"   Learning rate: {lr}")
    print(f"   Early stopping patience: {patience}")
    print(f"   Model save path: {save_path}")
    
    # Create models directory if it doesn't exist
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    # Loss function and optimizer
    criterion = rmse_loss_function  # Use RMSE as loss function
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=8, min_lr=1e-6
    )
    
    # Training tracking
    best_val_rmse = float('inf')
    patience_counter = 0
    history = {
        'train_loss': [], 'train_rmse': [], 'train_mae': [],
        'val_loss': [], 'val_rmse': [], 'val_mae': [],
        'learning_rates': []
    }
    
    print(f"\\nStarting training loop...")
    print("=" * 80)
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_predictions, train_targets = [], []
        
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1:2d}/{epochs} [Train]')
        for images, ages in train_pbar:
            images, ages = images.to(device), ages.to(device)
            
            optimizer.zero_grad()
            predictions = model(images).squeeze()
            loss = criterion(predictions, ages)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_predictions.extend(predictions.detach().cpu().numpy())
            train_targets.extend(ages.detach().cpu().numpy())
            
            # Update progress bar
            train_pbar.set_postfix({
                'Loss': f'{loss.item():.3f}',
                'LR': f'{optimizer.param_groups[0]["lr"]:.2e}'
            })
        
        # Calculate training metrics
        train_loss /= len(train_loader)
        train_rmse = calculate_rmse(train_predictions, train_targets)
        train_mae = calculate_mae(train_predictions, train_targets)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_predictions, val_targets = [], []
        
        with torch.no_grad():
            for images, ages in val_loader:
                images, ages = images.to(device), ages.to(device)
                predictions = model(images).squeeze()
                loss = criterion(predictions, ages)
                
                val_loss += loss.item()
                val_predictions.extend(predictions.cpu().numpy())
                val_targets.extend(ages.cpu().numpy())
        
        # Calculate validation metrics
        val_loss /= len(val_loader)
        val_rmse = calculate_rmse(val_predictions, val_targets)
        val_mae = calculate_mae(val_predictions, val_targets)
        
        # Update learning rate
        scheduler.step(val_rmse)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_rmse'].append(train_rmse)
        history['train_mae'].append(train_mae)
        history['val_loss'].append(val_loss)
        history['val_rmse'].append(val_rmse)
        history['val_mae'].append(val_mae)
        history['learning_rates'].append(current_lr)
        
        # Print epoch results
        print(f'Epoch {epoch+1:2d}: '
              f'Train RMSE: {train_rmse:5.2f} | Train MAE: {train_mae:5.2f} | '
              f'Val RMSE: {val_rmse:5.2f} | Val MAE: {val_mae:5.2f} | '
              f'LR: {current_lr:.2e}')
        
        # Save best model based on validation RMSE
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_rmse': val_rmse,
                'val_mae': val_mae,
                'train_rmse': train_rmse,
                'train_mae': train_mae,
                'history': history
            }, save_path)
            print(f'New best model saved! Val RMSE: {val_rmse:.2f} years')
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= patience:
            print(f'\\nEarly stopping triggered after {patience} epochs without improvement')
            break
        
        print("-" * 80)
    
    print(f'\\nTraining completed!')
    print(f'   Best validation RMSE: {best_val_rmse:.2f} years')
    print(f'   Total epochs: {epoch + 1}')
    print(f'   Model saved to: {save_path}')
    
    return history

print("RMSE-optimized training function defined")
print("   Uses RMSE as primary loss function")
print("   Tracks both RMSE and MAE metrics")
print("   Includes learning rate scheduling and early stopping")
print("   Saves best model based on validation RMSE")

In [ ]:
# Start training the model
print("Starting MobileNetV3 RMSE optimization training...")
print("="*80)

# Train the model
training_history = train_mobilenetv3_rmse(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=50,
    lr=0.001,
    patience=15,
    save_path="models/best_mobilenetv3_rmse.pth"
)

print("\\nTraining completed successfully!")
print(f"Final training metrics:")
if training_history['val_rmse']:
    print(f"   Best validation RMSE: {min(training_history['val_rmse']):.2f} years")
    print(f"   Best validation MAE: {min(training_history['val_mae']):.2f} years")
    print(f"   Total epochs trained: {len(training_history['val_rmse'])}")

## 8. Model Evaluation and RMSE Calculation

In [ ]:
def load_best_model(model_path="models/best_mobilenetv3_rmse.pth"):
    """Load the best trained model"""
    if os.path.exists(model_path):
        print(f"Loading best model from: {model_path}")
        
        # Create new model instance
        best_model = create_mobilenetv3_age_model(dropout_rate=0.3, hidden_size=128)
        
        # Load checkpoint with weights_only=False to handle PyTorch 2.6 security changes
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
        best_model.load_state_dict(checkpoint['model_state_dict'])
        
        print(f"Model loaded successfully!")
        print(f"   Best validation RMSE: {checkpoint['val_rmse']:.2f} years")
        print(f"   Best validation MAE: {checkpoint['val_mae']:.2f} years")
        print(f"   Trained for {checkpoint['epoch']} epochs")
        
        return best_model, checkpoint
    else:
        print(f"Error: Model file not found: {model_path}")
        print("   Please ensure the model has been trained and saved.")
        return None, None

def comprehensive_rmse_evaluation(model, val_loader, device):
    """
    Perform comprehensive RMSE evaluation with age group analysis
    """
    print("Performing comprehensive RMSE evaluation...")
    
    # Get basic evaluation metrics
    results = evaluate_model_rmse(model, val_loader, device)
    
    # Age group analysis
    predictions = results['predictions']
    targets = results['targets']
    errors = results['errors']
    
    # Define age groups
    age_groups = {
        'Young (0-20)': (0, 20),
        'Adult (21-40)': (21, 40),
        'Middle (41-60)': (41, 60),
        'Senior (61+)': (61, 100)
    }
    
    print(f"\nOverall Performance Metrics:")
    print(f"   RMSE: {results['rmse']:.2f} years")
    print(f"   MAE:  {results['mae']:.2f} years")
    print(f"   Mean Error: {results['mean_error']:.2f} ± {results['std_error']:.2f} years")
    print(f"   Median Error: {results['median_error']:.2f} years")
    print(f"   Max Error: {results['max_error']:.2f} years")
    
    # Age group analysis
    print(f"\nAge Group Analysis:")
    group_metrics = {}
    
    for group_name, (min_age, max_age) in age_groups.items():
        # Filter data for this age group
        mask = (targets >= min_age) & (targets <= max_age)
        if np.sum(mask) > 0:
            group_predictions = predictions[mask]
            group_targets = targets[mask]
            group_errors = errors[mask]
            
            group_rmse = calculate_rmse(group_predictions, group_targets)
            group_mae = calculate_mae(group_predictions, group_targets)
            
            group_metrics[group_name] = {
                'count': np.sum(mask),
                'rmse': group_rmse,
                'mae': group_mae,
                'mean_error': np.mean(group_errors),
                'std_error': np.std(group_errors)
            }
            
            print(f"   {group_name}: RMSE={group_rmse:.2f}, MAE={group_mae:.2f}, n={np.sum(mask)}")
    
    results['group_metrics'] = group_metrics
    return results

# Load and evaluate the best model
print("Loading and evaluating best MobileNetV3 model...")
best_model, checkpoint = load_best_model("models/best_mobilenetv3_rmse.pth")

if best_model is not None:
    # Perform comprehensive evaluation
    evaluation_results = comprehensive_rmse_evaluation(best_model, val_loader, device)
    
    print(f"\nModel Performance Summary:")
    print(f"   Final RMSE: {evaluation_results['rmse']:.2f} years")
    print(f"   Final MAE: {evaluation_results['mae']:.2f} years")
    print(f"   Evaluation completed on {len(evaluation_results['predictions'])} samples")
else:
    print("Error: Could not load model for evaluation")
    print("   Please run the training cell first to create the model")

## 9. Results Visualization

In [ ]:
def plot_training_history(history, save_path="plots/training_history.png"):
    """Plot training history with RMSE and MAE metrics"""
    
    # Create plots directory if it doesn't exist
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    epochs = range(1, len(history['train_rmse']) + 1)
    
    # RMSE plot
    axes[0, 0].plot(epochs, history['train_rmse'], 'b-', label='Train RMSE', linewidth=2)
    axes[0, 0].plot(epochs, history['val_rmse'], 'r-', label='Validation RMSE', linewidth=2)
    axes[0, 0].set_title('RMSE During Training', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('RMSE (years)')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # MAE plot
    axes[0, 1].plot(epochs, history['train_mae'], 'b-', label='Train MAE', linewidth=2)
    axes[0, 1].plot(epochs, history['val_mae'], 'r-', label='Validation MAE', linewidth=2)
    axes[0, 1].set_title('MAE During Training', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('MAE (years)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Loss plot
    axes[1, 0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[1, 0].plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    axes[1, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('RMSE Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning rate plot
    axes[1, 1].plot(epochs, history['learning_rates'], 'g-', linewidth=2)
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Training history plot saved to: {save_path}")

def plot_prediction_analysis(results, save_path="plots/prediction_analysis.png"):
    """Create comprehensive prediction analysis plots"""
    
    predictions = results['predictions']
    targets = results['targets']
    errors = results['errors']
    
    # Create plots directory if it doesn't exist
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Prediction vs Actual scatter plot
    axes[0, 0].scatter(targets, predictions, alpha=0.6, s=10)
    axes[0, 0].plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect Prediction')
    axes[0, 0].set_xlabel('Actual Age (years)')
    axes[0, 0].set_ylabel('Predicted Age (years)')
    axes[0, 0].set_title(f'Predictions vs Actual Ages\nRMSE: {results["rmse"]:.2f} years')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_aspect('equal')
    
    # Error distribution histogram
    axes[0, 1].hist(errors, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 1].axvline(np.mean(errors), color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: {np.mean(errors):.2f}')
    axes[0, 1].axvline(np.median(errors), color='orange', linestyle='--', linewidth=2, 
                      label=f'Median: {np.median(errors):.2f}')
    axes[0, 1].set_xlabel('Absolute Error (years)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].set_title('Error Distribution')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Error vs Age scatter plot
    axes[1, 0].scatter(targets, errors, alpha=0.6, s=10, color='orange')
    axes[1, 0].set_xlabel('Actual Age (years)')
    axes[1, 0].set_ylabel('Absolute Error (years)')
    axes[1, 0].set_title('Prediction Error vs Age')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Age group performance
    if 'group_metrics' in results:
        group_names = list(results['group_metrics'].keys())
        group_rmse = [results['group_metrics'][name]['rmse'] for name in group_names]
        group_mae = [results['group_metrics'][name]['mae'] for name in group_names]
        
        x = np.arange(len(group_names))
        width = 0.35
        
        axes[1, 1].bar(x - width/2, group_rmse, width, label='RMSE', alpha=0.8)
        axes[1, 1].bar(x + width/2, group_mae, width, label='MAE', alpha=0.8)
        axes[1, 1].set_xlabel('Age Groups')
        axes[1, 1].set_ylabel('Error (years)')
        axes[1, 1].set_title('Performance by Age Group')
        axes[1, 1].set_xticks(x)
        axes[1, 1].set_xticklabels(group_names, rotation=45, ha='right')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Prediction analysis plot saved to: {save_path}")

def create_rmse_summary_report(results, history, save_path="reports/rmse_summary.txt"):
    """Create a comprehensive RMSE performance report"""
    
    # Create reports directory if it doesn't exist
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    
    with open(save_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MobileNetV3 RMSE Optimization - Performance Summary Report\n")
        f.write("="*80 + "\n\n")
        
        # Overall metrics
        f.write("OVERALL PERFORMANCE METRICS:\n")
        f.write(f"   Root Mean Square Error (RMSE): {results['rmse']:.3f} years\n")
        f.write(f"   Mean Absolute Error (MAE): {results['mae']:.3f} years\n")
        f.write(f"   Mean Error: {results['mean_error']:.3f} ± {results['std_error']:.3f} years\n")
        f.write(f"   Median Error: {results['median_error']:.3f} years\n")
        f.write(f"   Maximum Error: {results['max_error']:.3f} years\n")
        f.write(f"   Total samples evaluated: {len(results['predictions']):,}\n\n")
        
        # Training summary
        if history:
            f.write("TRAINING SUMMARY:\n")
            f.write(f"   Total epochs: {len(history['val_rmse'])}\n")
            f.write(f"   Best validation RMSE: {min(history['val_rmse']):.3f} years\n")
            f.write(f"   Best validation MAE: {min(history['val_mae']):.3f} years\n")
            f.write(f"   Final learning rate: {history['learning_rates'][-1]:.2e}\n\n")
        
        # Age group analysis
        if 'group_metrics' in results:
            f.write("AGE GROUP ANALYSIS:\n")
            for group_name, metrics in results['group_metrics'].items():
                f.write(f"   {group_name}:\n")
                f.write(f"      RMSE: {metrics['rmse']:.3f} years\n")
                f.write(f"      MAE: {metrics['mae']:.3f} years\n")
                f.write(f"      Samples: {metrics['count']:,}\n")
                f.write(f"      Mean Error: {metrics['mean_error']:.3f} ± {metrics['std_error']:.3f}\n\n")
        
        # Error percentiles
        errors = results['errors']
        f.write("ERROR DISTRIBUTION:\n")
        percentiles = [10, 25, 50, 75, 90, 95, 99]
        for p in percentiles:
            f.write(f"   {p}th percentile: {np.percentile(errors, p):.2f} years\n")
    
    print(f"RMSE summary report saved to: {save_path}")

# Generate visualizations and reports
if 'evaluation_results' in locals() and 'training_history' in locals():
    print("Generating visualization and reports...")
    
    # Plot training history
    plot_training_history(training_history)
    
    # Plot prediction analysis
    plot_prediction_analysis(evaluation_results)
    
    # Create summary report
    create_rmse_summary_report(evaluation_results, training_history)
    
    print("All visualizations and reports generated successfully!")
    
else:
    print("Warning: Training history or evaluation results not available")
    print("   Please run the training and evaluation cells first")

## Summary

This notebook provides **RMSE-optimized MobileNetV3** training and evaluation for age estimation:

### Key Features:
- **RMSE-focused optimization**: Uses RMSE as primary loss function and evaluation metric
- **Comprehensive evaluation**: Age group analysis and detailed performance metrics  
- **Advanced training pipeline**: Early stopping, learning rate scheduling, best model saving
- **Detailed visualizations**: Training curves, prediction analysis, error distribution
- **Performance reporting**: Automated report generation with detailed statistics

### Model Architecture:
- **Backbone**: MobileNetV3-small (pretrained on ImageNet)
- **Regression Head**: 3-layer fully connected network with ReLU and Dropout
- **Output**: Single continuous age prediction (0-100+ years)
- **Optimization**: AdamW optimizer with ReduceLROnPlateau scheduling

### Dataset:
- **UTKFace**: 48,212 face images with age labels
- **Split**: 80% training, 20% validation
- **Augmentation**: Random crop, flip, color jitter, rotation
- **Preprocessing**: ImageNet normalization

### Evaluation Metrics:
- **Primary**: Root Mean Square Error (RMSE)
- **Secondary**: Mean Absolute Error (MAE)
- **Analysis**: Age group performance, error distribution, percentiles

### Results Output:
- **Model**: Best model saved based on validation RMSE
- **Plots**: Training history, prediction scatter plots, error analysis
- **Report**: Comprehensive performance summary with statistics

### Perfect for:
- **RMSE optimization**: Focus on minimizing root mean square error
- **Age estimation research**: Detailed performance analysis
- **Model comparison**: Standardized evaluation framework
- **Performance benchmarking**: Comprehensive metrics and visualizations